In [1]:
!pip install --quiet qiskit
!pip install --quiet mqt.bench

!wget https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb  -O packages-microsoft-prod.deb
!dpkg -i packages-microsoft-prod.deb
!apt-get update
!apt-get install -y dotnet-sdk-8.0

!pip install --quiet qsharp azure-quantum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 11.9 MB/s eta 0:00:00
--2026-04-16 22:50:43--  https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb
Resolving packages.microsoft.com (packages.microsoft.com)... 13.107.213.38, 13.107.246.38, 2620:1ec:46::38, ...
Connecting to packages.microsoft.com (packages.microsoft.com)|13.107.213.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3690 (3.6K) [application/octet-stream]
Saving to: ‘packages-microsoft-prod.deb’

packages-microsoft- 100%[===================>]   3.60K  --.-KB/s    in 0s      

2026-04-16 22:50:43 (1.51 GB/s) - ‘packages-microsoft-prod.deb’ saved [3690/3690]

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qsharp.estimator import EstimatorParams, ErrorBudgetPartition, LogicalCounts
from qsharp.interop.qiskit import estimate
from mqt.bench import get_benchmark, BenchmarkLevel
from scipy.optimize import minimize_scalar
import sys
from io import StringIO

QISKIT_STD_GATES = [
    "p", "x", "y", "z", "h", "s", "sdg", "t", "tdg", "sx",
    "rx", "ry", "rz", "cx", "cy", "cz", "cp", "crx", "cry",
    "crz", "ch", "ccx", "cu", "id", "u1", "u2", "u3"
]


# ──────────────────────────────────────────────────────────────────────────────
# Resource Estimation & Cost 
# ──────────────────────────────────────────────────────────────────────────────

def estimate_resources(logical_counts, error_budget_dist):
    """Estimates physical resources for a given error budget distribution."""
    params = EstimatorParams()
    params.error_budget = ErrorBudgetPartition()
    params.error_budget.logical = error_budget_dist['logical']
    params.error_budget.t_states = error_budget_dist['t_states']
    params.error_budget.rotations = error_budget_dist['rotations']

    result = logical_counts.estimate(params=params)

    physical_qubits = result["physicalCounts"]["physicalQubits"]
    runtime = result["physicalCounts"]["runtime"]

    return {
        'physical_qubits': physical_qubits,
        'runtime': runtime,
        'metric': runtime * physical_qubits
    }


def compute_cost(qubits, runtime, weight_qubits=0.5):
    """Compute cost with proper normalization."""
    q = max(qubits, 1)
    r = max(runtime, 1)
    cost = (q ** weight_qubits) * (r ** (1 - weight_qubits))
    return cost


def evaluate_distribution(logical_counts, budgets, total_budget, weight_qubits=0.5):
    """Evaluate cost for given budget fractions (budgets must sum to 1)."""
    dist = {
        'logical': budgets[0] * total_budget,
        't_states': budgets[1] * total_budget,
        'rotations': budgets[2] * total_budget
    }
    result = estimate_resources(logical_counts, dist)
    return compute_cost(result['physical_qubits'], result['runtime'], weight_qubits)


# ──────────────────────────────────────────────────────────────────────────────
# DEFINITION 1 (Game Formulation)
# ──────────────────────────────────────────────────────────────────────────────
#
# We define a three-player noncooperative game G = (N, S, C) where:
#
#   N = {1, 2, 3}  — players representing Logical, T-state, and Rotation
#                     error budget components
#
#   S = { s = (s_1, s_2, s_3) ∈ R^3 | s_i >= ε, Σ s_i = 1 }
#       — the joint strategy space is the ε-interior of the 2-simplex Δ²
#       — each s_i is the fraction of total error budget ε_total allocated
#         to component i
#       — ε > 0 is a small constant ensuring feasibility (each component
#         receives a nonzero budget)
#
#   C(s) = Q(s)^w · T(s)^(1-w)  — the shared cost function
#       — Q(s) = physical qubits required under allocation s
#       — T(s) = runtime (in microseconds) under allocation s
#       — w ∈ (0,1) is a weighting parameter (default w = 0.5)
#
# KEY PROPERTY: Since all three players share the SAME cost function C(s),
# this is a common-interest game [Aumann & Sorin, 1989]. This has two
# critical consequences:
#
#   (i)  The game is an exact potential game with potential Φ = C.
#        IBR therefore converges to a pure Nash equilibrium [Monderer &
#        Shapley, 1996].
#
#   (ii) Every Nash equilibrium that minimises C is Pareto optimal.
#        Proof: Suppose s* minimises C and is not Pareto optimal. Then
#        ∃ s' with C_i(s') ≤ C_i(s*) ∀i, strict for some i. But
#        C_i = C ∀i, so C(s') < C(s*), contradicting minimality. □
#
# Therefore, we do NOT need separate runtime checks for Nash or Pareto —
# these are guaranteed by the algorithm's mathematical properties.
#
# ──────────────────────────────────────────────────────────────────────────────


# ──────────────────────────────────────────────────────────────────────────────
# ALGORITHM 1: Best Response Operator
# ──────────────────────────────────────────────────────────────────────────────
#
# Definition (Best Response): For player i, given opponents' strategies s_{-i},
#
#   BR_i(s_{-i}) = argmin_{s_i ∈ [ε, 1 - Σ_{j≠i} s_j]}  C(s_i, s_{-i})
#
# On the simplex with budget constraint Σ s_i = 1, when player i varies its
# allocation s_i, the remaining budget (1 - s_i) must be distributed among
# the other two players. We hold their RELATIVE RATIO fixed (i.e., they do
# not react), which is the standard approach for simplex-constrained games.
#
# This reduces to a 1D optimisation problem, solved via Brent's method.
# ──────────────────────────────────────────────────────────────────────────────

# Minimum allocation per player — ensures feasibility of the resource estimator
EPSILON = 0.005

def best_response(logical_counts, current_budgets, player_idx,
                  total_budget, weight_qubits=0.5):
    """
    Compute best response for player `player_idx`.

    Given current strategy profile s on the simplex, player i optimises its
    allocation while opponents' relative shares remain fixed. The simplex
    constraint is maintained by redistributing 1 - s_i proportionally.

    Parameters
    ----------
    logical_counts : LogicalCounts
        Resource estimation oracle.
    current_budgets : np.ndarray, shape (3,), sums to 1
        Current strategy profile.
    player_idx : int in {0, 1, 2}
        Index of the optimising player.
    total_budget : float
        Total error budget (e.g., 0.1).
    weight_qubits : float
        Cost function weight parameter.

    Returns
    -------
    new_budgets : np.ndarray, shape (3,), sums to 1
        Updated strategy profile after player i's best response.
    best_cost : float
        Cost at the new profile.
    """
    others = [j for j in range(3) if j != player_idx]
    j, k = others

    # Fix opponents' relative ratio
    other_total = current_budgets[j] + current_budgets[k]
    if other_total < 1e-12:
        ratio_j = 0.5
    else:
        ratio_j = current_budgets[j] / other_total

    lo = EPSILON
    hi = 1.0 - 2.0 * EPSILON

    def cost_fn(x_i):
        residual = 1.0 - x_i
        b = np.empty(3)
        b[player_idx] = x_i
        b[j] = ratio_j * residual
        b[k] = (1.0 - ratio_j) * residual
        if b[j] < EPSILON or b[k] < EPSILON:
            return 1e30
        return evaluate_distribution(logical_counts, b, total_budget, weight_qubits)

    result = minimize_scalar(cost_fn, bounds=(lo, hi), method='bounded',
                             options={'xatol': 1e-4, 'maxiter': 30})

    x_star = result.x
    residual = 1.0 - x_star
    new_budgets = np.empty(3)
    new_budgets[player_idx] = x_star
    new_budgets[j] = ratio_j * residual
    new_budgets[k] = (1.0 - ratio_j) * residual

    return new_budgets, result.fun


# ──────────────────────────────────────────────────────────────────────────────
# ALGORITHM 2: Iterated Best Response (IBR)
# ──────────────────────────────────────────────────────────────────────────────
#
# Theorem [Monderer & Shapley, 1996]: In a finite exact potential game,
# iterated best response converges to a pure Nash equilibrium.
#
# Our game has potential Φ = C (the shared cost). Each best response
# strictly decreases Φ (or leaves it unchanged at equilibrium). Since Φ
# is bounded below by 0, the sequence {Φ(s^(t))} converges.
#
# Algorithm:
#   Input:  Initial profile s^(0), tolerance δ > 0
#   For t = 0, 1, 2, ...:
#       For each player i ∈ {1, 2, 3}:
#           s_i^(t+1) ← BR_i(s_{-i}^(t))         // best response step
#       If max_i |C(s^(t+1)) - C(s^(t))| < δ:     // ε-Nash condition
#           Return s^(t+1)                          // Nash equilibrium
#
# ε-Nash condition: No player can reduce cost by more than δ through
# unilateral deviation. At convergence, s* is a pure Nash equilibrium.
#
# By Property (ii) above, if s* is cost-minimising among all equilibria,
# it is also Pareto optimal.
# ──────────────────────────────────────────────────────────────────────────────

def nash_equilibrium_optimization(logical_counts, total_budget,
                                  max_iterations=200,
                                  tolerance=1e-6,
                                  weight_qubits=0.5,
                                  verbose=False):
    """
    Find Nash equilibrium via Iterated Best Response with random restarts.

    Multiple random initialisations on the simplex are used to mitigate
    convergence to suboptimal equilibria. The equilibrium with the lowest
    cost is returned (guaranteed Pareto optimal by the common-interest
    game property).

    Parameters
    ----------
    logical_counts : LogicalCounts
        Resource estimation oracle.
    total_budget : float
        Total error budget.
    max_iterations : int
        Maximum IBR iterations per restart.
    tolerance : float
        ε-Nash convergence threshold.
    weight_qubits : float
        Cost function weight.
    verbose : bool
        Print convergence information.

    Returns
    -------
    dict with keys: name, logical, t_states, rotations, iterations, converged
    """
    if verbose:
        print(f"\n{'='*70}")
        print("Nash Equilibrium via Iterated Best Response")
        print(f"{'='*70}")

    # ── Random restarts on the simplex ──
    # Dirichlet(1,1,1) is the uniform distribution over the simplex.
    # We use a small number of restarts (5) to avoid suboptimal equilibria
    # without significant computational overhead.
    n_restarts = 5
    rng = np.random.default_rng(seed=42)
    init_points = [np.array([1/3, 1/3, 1/3])]  # always include uniform
    for _ in range(n_restarts - 1):
        raw = rng.dirichlet([1.0, 1.0, 1.0])
        raw = np.clip(raw, EPSILON, None)
        raw /= raw.sum()
        init_points.append(raw)

    # ── Baseline (uniform) ──
    baseline_cost = evaluate_distribution(
        logical_counts, np.array([1/3, 1/3, 1/3]), total_budget, weight_qubits
    )

    if verbose:
        print(f"Baseline cost (uniform): {baseline_cost:.6e}")
        print(f"Running IBR from {n_restarts} random initialisations...")

    global_best_budgets = np.array([1/3, 1/3, 1/3])
    global_best_cost = baseline_cost
    global_best_iters = 0
    global_converged = False

    # ── IBR from each starting point ──
    for start_idx, init_budgets in enumerate(init_points):
        current = init_budgets.copy()
        current_cost = evaluate_distribution(
            logical_counts, current, total_budget, weight_qubits
        )
        converged = False

        for iteration in range(max_iterations):
            max_improvement = 0.0

            # Cyclic best response: player 0 -> 1 -> 2
            for player_idx in range(3):
                new_budgets, new_cost = best_response(
                    logical_counts, current, player_idx,
                    total_budget, weight_qubits
                )
                improvement = current_cost - new_cost
                if improvement > 0:
                    current = new_budgets
                    current_cost = new_cost
                max_improvement = max(max_improvement, abs(improvement))

            # ε-Nash: no player can improve by more than tolerance
            if max_improvement < tolerance:
                converged = True
                break

        # Track best across restarts
        if current_cost < global_best_cost:
            global_best_cost = current_cost
            global_best_budgets = current.copy()
            global_best_iters = iteration + 1
            global_converged = converged

        if verbose:
            imp = ((baseline_cost - current_cost) / baseline_cost) * 100
            print(f"  Restart {start_idx+1}: cost={current_cost:.6e} "
                  f"({imp:+.2f}%), iters={iteration+1}, converged={converged}")

    if verbose:
        imp = ((baseline_cost - global_best_cost) / baseline_cost) * 100
        print(f"\n{'='*70}")
        print(f"Nash Equilibrium Found")
        print(f"{'='*70}")
        print(f"Cost improvement: {imp:+.2f}% vs uniform")
        print(f"Budgets: L={global_best_budgets[0]:.4f}, "
              f"T={global_best_budgets[1]:.4f}, "
              f"R={global_best_budgets[2]:.4f}")

    return {
        'name': 'Nash_Equilibrium_IBR',
        'logical': global_best_budgets[0] * total_budget,
        't_states': global_best_budgets[1] * total_budget,
        'rotations': global_best_budgets[2] * total_budget,
        'iterations': global_best_iters,
        'converged': global_converged
    }


# ──────────────────────────────────────────────────────────────────────────────
# Circuit Processing Pipeline 
# ──────────────────────────────────────────────────────────────────────────────

def process_single_circuit(benchmark_name, circuit_size, total_error_budget=0.1,
                           max_iterations=200, tolerance=1e-6, verbose=False):
    """Process a single circuit and return results."""
    try:
        # Load benchmark
        qc = get_benchmark(
            benchmark=benchmark_name,
            level=BenchmarkLevel.ALG,
            circuit_size=circuit_size
        )

        # Transpile
        transpiled_qc = transpile(qc, basis_gates=QISKIT_STD_GATES, optimization_level=3)

        # Estimate logical counts
        estimation_result = estimate(transpiled_qc)
        logical_counts_dict = estimation_result["logicalCounts"]
        logical_counts = LogicalCounts(logical_counts_dict)

        # Run Nash Equilibrium optimization
        if not verbose:
            old_stdout = sys.stdout
            sys.stdout = StringIO()

        optimal_distribution = nash_equilibrium_optimization(
            logical_counts,
            total_error_budget,
            max_iterations=max_iterations,
            tolerance=tolerance,
            weight_qubits=0.5,
            verbose=verbose
        )

        if not verbose:
            sys.stdout = old_stdout

        # Evaluate optimal
        optimal_result = estimate_resources(logical_counts, optimal_distribution)

        # Baseline - uniform distribution
        uniform_value = total_error_budget / 3
        uniform_distribution = {
            'logical': uniform_value,
            't_states': uniform_value,
            'rotations': uniform_value
        }
        uniform_result = estimate_resources(logical_counts, uniform_distribution)

        # Calculate improvements
        qubit_imp = ((uniform_result['physical_qubits'] - optimal_result['physical_qubits'])
                     / uniform_result['physical_qubits']) * 100
        runtime_imp = ((uniform_result['runtime'] - optimal_result['runtime'])
                       / uniform_result['runtime']) * 100
        metric_imp = ((uniform_result['metric'] - optimal_result['metric'])
                      / uniform_result['metric']) * 100

        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'original_depth': qc.depth(),
            'transpiled_depth': transpiled_qc.depth(),
            'logical_qubits': logical_counts_dict['numQubits'],
            't_gates': logical_counts_dict['tCount'],
            'rotations': logical_counts_dict['rotationCount'],
            'rotation_depth': logical_counts_dict['rotationDepth'],
            'ccz_gates': logical_counts_dict['cczCount'],
            # Uniform distribution results
            'uniform_physical_qubits': uniform_result['physical_qubits'],
            'uniform_runtime': uniform_result['runtime'],
            'uniform_metric': uniform_result['metric'],
            # Optimal distribution budgets
            'optimal_logical_budget': optimal_distribution['logical'],
            'optimal_t_states_budget': optimal_distribution['t_states'],
            'optimal_rotations_budget': optimal_distribution['rotations'],
            # Optimal distribution results
            'optimal_physical_qubits': optimal_result['physical_qubits'],
            'optimal_runtime': optimal_result['runtime'],
            'optimal_metric': optimal_result['metric'],
            # Improvements
            'qubit_improvement_pct': qubit_imp,
            'runtime_improvement_pct': runtime_imp,
            'metric_improvement_pct': metric_imp,
            'nash_iterations': optimal_distribution.get('iterations', 0),
            'status': 'success'
        }

    except Exception as e:
        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'status': 'failed',
            'error': str(e)
        }


def main():
    import pandas as pd
    from datetime import datetime

    benchmarks_config = [
        # Quantum State Preparation
        ("ghz", [3, 4, 5, 6, 7, 8, 9, 10, 15, 16, 17, 18, 19, 20,21,22,23,24, 25, 30]),
        ("graphstate", [3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40]),
        ("wstate", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),

        # Quantum Algorithms
        ("qft", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,25,30]),
        ("qftentangled", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
        ("qpeexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("qpeinexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("dj", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),
        ("bv", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20,21,22,23,24, 25, 30]),
        ("grover", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),

        # Variational Algorithms
        ("qaoa", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),

        # Amplitude Estimation
        ("ae", [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),

        # Quantum Chemistry
        ("hhl", [5, 7, 9, 11, 13, 15,17]),

        # Machine Learning
        ("qnn", [4, 5, 6, 7, 8, 9, 10,11,12,13,14,15]),

        #New Addition
        ("cdkm_ripple_carry_adder", [ 4, 8,12,14,16,18,20,22,24,26]),
        ("full_adder",[4,8,12,14,16,18,20,22,24,26,28]),
        ("half_adder",[3,5,7,9,11,13,15,17,19,21,23,25,27]),
        ("hrs_cumulative_multiplier",[5,9,13,17,21,25]),
        ("randomcircuit",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("vqe_su2",[2,3,4,5,6,7,8,9,10,15,20,25,30,35,40,80]),
        ("vqe_two_local",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("qwalk",[4,5,6,7,8,9,10,11,12,13,14,15]),

        ("shor", [18,42]),
        ("modular_adder",[2,4,6,8,10,12,14,16,18,20,22,24,26,28,30]),
        ("vqe_real_amp",[2,3,4,5,6,7,8,9,10,11,12,13,14,15,35,40,45,50,55,60]),
        ("bmw_quark_cardinality",[2,3,4,5,10,11,12,13,14,15,40,60]),
        ("bmw_quark_copula",[2,4,6,8,10,12,14,16,18,20,28,30,50,60]),
        ("rg_qft_multiplier",[4,8,12,16,24,28,32]),
        ("draper_qft_adder",[2,4,6,8,10,14,16,18,20,22,24,26,28,30,40,50,60,90]),
        ("multiplier",[4,8,12,16,20,24,28,32,36,40,44,48,52,60]),
        ("vbe_ripple_carry_adder",[4,7,10,13,16,19,22,25,31,34,37,40,43,46,61,91]),
    ]

    print("="*70)
    print("MQT Benchmark Suite - Nash Equilibrium Resource Optimization")
    print("="*70)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nMethodology: Iterated Best Response -> Pure Nash Equilibrium")
    print(f"Game type: Common-interest potential game on the 2-simplex")
    print(f"Convergence: Guaranteed by Monderer & Shapley (1996)")
    print(f"Pareto optimality: Guaranteed by common-interest property")
    print(f"Total benchmarks: {len(benchmarks_config)}")
    total_circuits = sum(len(sizes) for _, sizes in benchmarks_config)
    print(f"Total circuits to process: {total_circuits}")
    print("="*70)

    all_results = []
    circuit_count = 0

    for benchmark_name, circuit_sizes in benchmarks_config:
        for circuit_size in circuit_sizes:
            circuit_count += 1
            print(f"\nProcessing [{circuit_count}/{total_circuits}]: "
                  f"{benchmark_name} (size={circuit_size})...", end=" ")

            result = process_single_circuit(
                benchmark_name=benchmark_name,
                circuit_size=circuit_size,
                total_error_budget=0.1,
                max_iterations=200,
                tolerance=1e-6,
                verbose=False
            )

            all_results.append(result)

            if result['status'] == 'success':
                print(f"✓ (Metric: {result['metric_improvement_pct']:+.2f}%, "
                      f"Iters: {result['nash_iterations']})")
            else:
                print(f"✗ (Error: {result.get('error', 'Unknown')})")

    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f'nash_equilibrium_results_{timestamp}.csv'
    df.to_csv(csv_filename, index=False)

    print(f"\n{'='*70}")
    print(f"Results saved to: {csv_filename}")
    print(f"{'='*70}")

    successful_results = df[df['status'] == 'success']
    failed_count = len(df) - len(successful_results)

    if len(successful_results) > 0:
        avg_qubit_imp = successful_results['qubit_improvement_pct'].mean()
        avg_runtime_imp = successful_results['runtime_improvement_pct'].mean()
        avg_metric_imp = successful_results['metric_improvement_pct'].mean()

        print(f"\n{'='*70}")
        print("OVERALL RESULTS - NASH EQUILIBRIUM vs UNIFORM DISTRIBUTION")
        print(f"{'='*70}")
        print(f"\nTotal circuits processed: {len(df)}")
        print(f"  ✓ Successful: {len(successful_results)}")
        print(f"  ✗ Failed: {failed_count}")

        print(f"\n{'-'*70}")
        print("AVERAGE IMPROVEMENTS:")
        print(f"{'-'*70}")
        print(f"Physical Qubits:  {avg_qubit_imp:+.2f}%")
        print(f"Runtime:          {avg_runtime_imp:+.2f}%")
        print(f"Overall Metric:   {avg_metric_imp:+.2f}%")

        best_metric = successful_results.loc[successful_results['metric_improvement_pct'].idxmax()]

        print(f"\n{'-'*70}")
        print("BEST PERFORMING CIRCUIT:")
        print(f"{'-'*70}")
        print(f"{best_metric['benchmark']} (size={int(best_metric['circuit_size'])})")
        print(f"  Metric improvement: {best_metric['metric_improvement_pct']:+.2f}%")
        print(f"  Allocations: L={best_metric['optimal_logical_budget']:.4f}, "
              f"T={best_metric['optimal_t_states_budget']:.4f}, "
              f"R={best_metric['optimal_rotations_budget']:.4f}")

    print(f"\n{'='*70}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

MQT Benchmark Suite - Nash Equilibrium Resource Optimization
Start time: 2026-04-16 22:51:29

Methodology: Iterated Best Response -> Pure Nash Equilibrium
Game type: Common-interest potential game on the 2-simplex
Convergence: Guaranteed by Monderer & Shapley (1996)
Pareto optimality: Guaranteed by common-interest property
Total benchmarks: 31
Total circuits to process: 433

Processing [1/433]: ghz (size=3)... ✓ (Metric: +0.00%, Iters: 0)

Processing [2/433]: ghz (size=4)... ✓ (Metric: +0.00%, Iters: 0)

Processing [3/433]: ghz (size=5)... ✓ (Metric: +0.00%, Iters: 0)

Processing [4/433]: ghz (size=6)... ✓ (Metric: +78.40%, Iters: 200)

Processing [5/433]: ghz (size=7)... ✓ (Metric: +78.40%, Iters: 200)

Processing [6/433]: ghz (size=8)... ✓ (Metric: +78.40%, Iters: 200)

Processing [7/433]: ghz (size=9)... ✓ (Metric: +78.40%, Iters: 200)

Processing [8/433]: ghz (size=10)... ✓ (Metric: +78.40%, Iters: 200)

Processing [9/433]: ghz (size=15)... ✓ (Metric: +0.00%, Iters: 0)

Processing 